In [ ]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = True

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/Jahdejosefina/projecto_ragTP2.git"
    REPO_BRANCH = ""
    REPO_DIR = "projecto_rag"
    DRIVE = "/content/drive/MyDrive/projecto_rag"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

Move its contents into /content/drive/MyDrive/projecto_rag/artifacts , delete it, then re-run this cell.
Colab bootstrap OK. Working directory: /content/projecto_rag


# Etapa 6 · Memoria entre sesiones

**Para qué.** Entre sesiones no sobrevive nada que no se escriba. Acá la memoria es un corpus que el
sistema escribe en tiempo de ejecución, a partir de su propia extracción (que puede equivocarse), y que
tiene que mantener actualizado.

**Entradas:** `eval/scenarios.jsonl` congelado (Etapa 2), el índice BM25 de la Etapa 3 (el recuperador
elegido en `stage3_selected_retriever.json`) y el loop de la Etapa 5: recuperar → ensamblar con
`grounded_stage4` → responder con `GroundedAnswer`. No modifica ninguno de esos insumos.
**Salidas:** `artifacts/<profile>/memory/stage6_<hash>/` (el almacén y su log de operaciones, por
escenario y por sesión) y `artifacts/<profile>/results/stage6_<hash>/` (resultados por sonda y brazo,
tablas por habilidad, auditoría, informe de corrida).

**Vocabulario CoALA:** la ventana ensamblada es la memoria de trabajo; las transcripciones reproducidas
son memoria episódica; los hechos extraídos son memoria semántica (y episódica, para eventos como una
compra); las preferencias sobre cómo responder son memoria procedimental.

**Brazos, sobre las mismas sondas:**

| Brazo | Qué recibe el generador además de los pasajes |
|---|---|
| `no_memory` | nada: el piso |
| `full_replay` | todas las sesiones pasadas, enteras: la línea de base honesta ("todo lo que entre") |
| `memory` | el espacio de memoria: recuerdos recuperados del almacén |
| `memory_filter` | lo mismo, y la recuperación del corpus filtrada por la norma que el usuario pidió usar (en todas las sondas: el sistema no sabe de antemano cuáles necesitan el corpus) |
| `replay_50`, `replay_25` | la reproducción con un presupuesto de historial menor: dónde deja de entrar |

## 1. Entorno y decisiones fijadas antes de medir

| Decisión | Valor | Por qué |
|---|---|---|
| `num_ctx` / `num_predict` | 8.192 / 700 | los mismos de las Etapas 4 y 5; tope del prompt 7.428 |
| Pasajes | 1.500 tokens | igual en todos los brazos: la diferencia entre brazos es solo memoria/historial |
| Espacio de memoria | 600 tokens, hasta 6 recuerdos | §7.3: 300-1.000 tokens, 3-10 recuerdos |
| Historial (reproducción) | hasta 3.000 tokens | los escenarios ocupan ~300: la reproducción completa siempre entra |
| Barrido | 50% y 25% del historial completo del escenario | presupuesto fijo, historial que crece: ahí deja de entrar |
| Recencia | 0,995 por hora sobre las fechas SIMULADAS | la sonda ocurre 24 h después de `after_session` |
| Consulta al corpus | la pregunta de la sonda, tal cual | igual en todos los brazos; la sesión nueva no tiene historial que reescribir |
| Veredicto sí/no | primera palabra de la respuesta | la misma corrección final de la Etapa 4 |
| Embedder de memoria | `BAAI/bge-m3`, el de la Etapa 3 | "con el mismo embedder" |

In [ ]:
import environment  # primero: mismas guardas del proyecto (JAX, caché de HF)
from dataclasses import asdict
from pathlib import Path
import ast
import csv
import hashlib
import json
import shutil
import sys
import time
import types
import warnings

SMOKE_TEST = False
SPLIT = 'test'               # 'test' solo cuando las decisiones estén fijadas
PROFILE = 'smoke' if SMOKE_TEST else 'full'
MODEL = 'qwen3:1.7b' if SMOKE_TEST else 'qwen3:8b'
RUN_SWEEP = not SMOKE_TEST  # brazos replay_50 / replay_25 (dónde deja de entrar la reproducción)
EMBEDDER = 'bge-m3'         # 'bge-m3' (el de la Etapa 3) o 'lexical' (respaldo declarado, sin GPU)
STAGE3_INDEX_DIR = None     # opcional: Path('/content/drive/MyDrive/.../index/stage3_xxxx') si el índice está en otro lado


ART = environment.artifacts_dir(SMOKE_TEST)          # resultados de esta corrida
ART_FULL = environment.artifacts_dir(False)          # corpus e índice: siempre los completos
DOC_PATH = ART_FULL / 'documents.jsonl'
EVAL_DIR = Path('eval')
DRIVE_ART = Path('/content/drive/MyDrive/projecto_rag/artifacts')

assert SPLIT in ('dev', 'test')
assert not (SPLIT == 'test' and SMOKE_TEST), 'Nunca evaluar test con smoke.'
if environment.on_colab() and not environment.ARTIFACTS.is_symlink():
    print('AVISO: artifacts/ no está enlazado a Drive. Se leen el corpus y el índice desde Drive si faltan, '
          'y al final los resultados se copian a Drive.')
environment.summary()

AVISO: artifacts/ no está enlazado a Drive. Se leen el corpus y el índice desde Drive si faltan, y al final los resultados se copian a Drive.
ENVIRONMENT
os             : Linux 6.6.159+
python         : 3.13.16
interpreter    : /usr/bin/python3
colab          : True
logical_cores  : 2
torch          : 2.11.0+cu130
cuda_available : True
gpu            : Tesla T4
gpu_memory_gb  : 15.64
ram_total_gb   : 13.6
ram_free_gb    : 2.1
HF_HOME        : /content/drive/MyDrive/projecto_rag/hf_cache
jax_guard      : True
ca_bundle      : not present (normal)
artifacts      : /content/projecto_rag/artifacts (local disk)


## 2. Funciones de la etapa

En el notebook, esta celda es una copia de `memory_stage6.py` (la genera `build_stage6_notebook.py`).
Importa de `grounded_stage4.py` el ensamblador, el schema de respuesta y la contención de la clave,
sin modificarlo. El hash del código forma parte de la procedencia de la corrida.

In [ ]:
"""Etapa 6: memoria entre sesiones.

El notebook notebooks/06_memory incluye una copia de este módulo (como las Etapas 3 y 4),
así puede abrirse solo en Colab. Reusa sin modificarlo `grounded_stage4.py` (el ensamblador,
el schema de respuesta y la contención de la clave), igual que el loop de la Etapa 5.

Qué contiene:
  - Esquemas: CandidateFact / Extraction (camino de escritura), WriteDecision, MemoryRecord.
    En todos, el razonamiento se declara primero y su nombre también queda primero
    alfabéticamente ("analysis" < ...).
  - MemoryStore: el almacén. Invalidá, no borres: un UPDATE cierra el registro viejo
    (valid_to) y agrega uno nuevo; nunca se elimina nada. Todo queda en un log de operaciones.
  - write_session: extraer hechos con un schema -> recuperar los recuerdos parecidos ->
    decidir ADD / UPDATE / NOOP / INVALIDATE -> guardar created_at, valid_from, valid_to.
    Solo se guardan los campos `text` de los hechos: nunca el `analysis` del modelo.
  - read_memories: relevancia (embedder) + recencia (timestamps SIMULADOS) + importancia,
    con min-max e informe del rango de cada término antes de normalizar.
  - memory_filter: los hechos del usuario que coinciden con metadatos del corpus (doc_id,
    fecha de publicación) se convierten en filtros de la recuperación.
  - build_probe_prompt: el prompt de una sonda con un espacio de memoria y/o historial,
    dentro de presupuestos declarados y del tope num_ctx - num_predict - margen.
  - score_probe, audit_records, ability_table: métricas sin juez.

No modifica la evaluación congelada ni los artefactos de etapas anteriores.
"""
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Callable, Literal
import json
import re
import unicodedata

from pydantic import BaseModel, Field

from grounded_stage4 import GroundedAnswer, assemble, contains_key, passage_block, render_passages


# --- configuración ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Stage6Config:
    """Todo lo que decide qué significa una fila de resultados (va al run_spec)."""
    model: str = "qwen3:8b"
    num_ctx: int = 8192              # fijo, igual que en las Etapas 4 y 5
    num_predict: int = 700           # reserva de salida de la respuesta a la sonda
    safety_margin: int = 64
    passage_budget: int = 1500       # pasajes del corpus, en tokens de Qwen3
    memory_budget: int = 600         # espacio de memoria (§7.3: 300-1.000)
    history_budget: int = 3000       # reproducción: "todo lo que entre" dentro de este tope
    memory_k: int = 6                # recuerdos por sonda (§7.3: 3-10)
    recency_decay: float = 0.995     # por hora, sobre timestamps simulados (§7.3)
    probe_offset_hours: int = 24     # la sonda se hace en una sesión nueva, un día después
    write_candidates: int = 3        # recuerdos parecidos que ve el decisor
    write_similarity: float = 0.55   # por debajo (y con otro subject) se agrega sin consultar al LLM
    extract_num_predict: int = 800
    decide_num_predict: int = 300
    sweep_fractions: tuple = (0.5, 0.25)
    temperature: float = 0.0
    seed: int = 0

    def __post_init__(self):
        if self.num_ctx <= self.num_predict:
            raise ValueError("num_ctx tiene que ser mayor que num_predict.")
        if self.passage_budget + self.memory_budget >= self.prompt_limit:
            raise ValueError("Pasajes + memoria no dejan lugar para sistema y pregunta.")
        if self.temperature != 0.0:
            raise ValueError("La etapa fija temperatura 0.")
        if not 0 < self.recency_decay < 1:
            raise ValueError("recency_decay en (0, 1).")

    @property
    def prompt_limit(self) -> int:
        return self.num_ctx - self.num_predict - self.safety_margin


# --- esquemas ----------------------------------------------------------------------------------
class CandidateFact(BaseModel):
    subject: str = Field(max_length=60, description="clave corta del atributo, snake_case")
    kind: Literal["semantic", "episodic", "procedural"]
    text: str = Field(max_length=240, description="afirmación autocontenida, solo lo que se dijo")
    importance: int = Field(ge=1, le=10)


class Extraction(BaseModel):
    # "analysis" < "facts": el razonamiento sale primero, y NUNCA se guarda.
    analysis: str = Field(max_length=400)
    facts: list[CandidateFact] = Field(max_length=8)


class WriteDecision(BaseModel):
    # El razonamiento declarado primero, obligatorio, y con un nombre que también quede primero
    # ("analysis" < "decision" < "target_id" < "text").
    analysis: str = Field(max_length=400)
    decision: Literal["ADD", "UPDATE", "NOOP", "INVALIDATE"]
    target_id: str | None = None
    text: str | None = None


class MemoryRecord(BaseModel):
    memory_id: str
    kind: Literal["semantic", "episodic", "procedural"]
    subject: str                       # un único valor ACTUAL por subject
    text: str
    created_at: datetime               # cuándo lo escribió el sistema (reloj real)
    valid_from: datetime               # cuándo pasó a ser cierto (tiempo SIMULADO de la sesión)
    valid_to: datetime | None = None   # se setea al invalidar; el registro se queda
    source_session: str
    importance: int = Field(default=5, ge=1, le=10)
    supersedes: str | None = None      # el registro que reemplazó (UPDATE)


def check_reasoning_first(*schemas) -> None:
    for s in schemas:
        props = list(s.model_json_schema()["properties"])
        if props[0] != "analysis" or sorted(props)[0] != "analysis":
            raise AssertionError(f"{s.__name__}: el razonamiento tiene que ir primero: {props}")


# --- prompts -------------------------------------------------------------------------------------
SYSTEM_EXTRACT = """Sos el módulo de memoria de un asistente que conversa con un mismo usuario a lo
largo de varias sesiones. Leé la transcripción de UNA sesión y extraé los hechos sobre el usuario
que valga la pena recordar en sesiones futuras.

Reglas:
- Solo lo que el usuario dijo o confirmó explícitamente en ESTA sesión. Nada de hipótesis,
  inferencias ni suposiciones ("probablemente", "parece").
- No omitas datos: cada nombre propio (tiendas, academias, productos), importe, fecha,
  canal e identificador de norma que el usuario mencione tiene que quedar en algún hecho,
  junto con lo que el usuario pidió hacer con él.
- Un hecho por atributo: no combines datos distintos en una misma oración.
- Cada hecho es una oración autocontenida en tercera persona ("El usuario ..."), con los datos
  exactos (nombres, importes, fechas, identificadores de normas) tal como se dijeron.
- Si el usuario corrige o cambia un dato, el hecho es el valor NUEVO, y conviene decir que
  reemplaza al anterior.
- subject: una clave corta en snake_case que identifique el atributo (por ejemplo
  canal_contacto, precio_compra, costo_envio, tienda_compra, norma_referencia,
  formato_respuesta). Si el atributo ya existe en la lista de subjects guardados, usá el mismo.
- kind: "procedural" para preferencias sobre cómo responder; "episodic" para eventos (una compra,
  un pago); "semantic" para el resto de los hechos sobre el usuario.
- importance: de 1 a 10.

Devolvé un JSON con analysis (breve) y facts."""

SYSTEM_DECIDE = """Sos el módulo de memoria de un asistente. Hay un hecho NUEVO extraído de la
última sesión y una lista de recuerdos YA GUARDADOS parecidos. Decidí qué hacer con el almacén:

- ADD: el hecho nuevo es información que no está guardada.
- UPDATE: el hecho nuevo da un valor distinto para el mismo atributo que un recuerdo guardado
  (un cambio o una corrección). target_id = el memory_id de ese recuerdo; text = el hecho nuevo.
  El recuerdo viejo no se borra: queda como "ya no vigente".
- NOOP: SOLO si un recuerdo guardado dice exactamente lo mismo (mismo atributo y mismo valor).
  Si el hecho nuevo agrega cualquier dato que no está guardado (un nombre, un lugar, un
  producto, una preferencia), es ADD.
- INVALIDATE: el usuario dijo que un recuerdo guardado dejó de ser cierto, sin dar un valor
  nuevo. target_id = el memory_id de ese recuerdo.

Devolvé un JSON con analysis (breve), decision, target_id y text."""

SYSTEM_PROBE = """Sos un asistente que conversa con un mismo usuario a lo largo de varias sesiones y
responde preguntas sobre normas argentinas de defensa del consumidor.

Podés recibir:
- MEMORIA: recuerdos guardados de sesiones anteriores, cada uno con su vigencia.
- HISTORIAL: los mensajes de sesiones anteriores.
- PASAJES: fragmentos del corpus normativo, delimitados por [doc_id · section_id].

Los datos del usuario salen solo de la MEMORIA o del HISTORIAL; la normativa sale solo de los
PASAJES. Si un dato cambió, el valor actual es el más reciente; si te preguntan por el valor
anterior, usá el que ya no está vigente. No inventes datos del usuario que no figuren.

Devolvé un objeto JSON con estos campos, en este orden:
- analysis: dos o tres oraciones sobre de dónde sale la respuesta.
- answer: la respuesta, breve y directa, en español. Copiá nombres, importes y fechas tal como
  figuran. Si la pregunta es de sí o no, empezá con "Sí" o "No". Si no tenés la información,
  escribí exactamente: "No tengo información suficiente para responder."
- can_answer: true solo si la memoria, el historial o los pasajes contienen la respuesta.
- citations: los section_id de los pasajes que usaste (lista vacía si respondiste solo con datos
  del usuario).
- verdict: "si" o "no" si la pregunta se responde con sí o no; "no_aplica" en otro caso."""


# --- utilidades de texto ---------------------------------------------------------------------------
def _norm(text: str) -> str:
    text = unicodedata.normalize("NFD", (text or "").lower())
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"(?<=\d)[.,\s](?=\d{3}(?:\D|$))", "", text)
    return " ".join(re.sub(r"[^\w\s]", " ", text).split())


def verdict_from_answer(answer: str) -> str | None:
    """'si'/'no' según la primera palabra de la respuesta, que el prompt obliga a ser Sí/No.

    Igual que la corrección final de la Etapa 4: el campo `verdict` del schema resultó
    inconsistente con el propio texto del modelo, así que se lee el texto pedido.
    """
    words = _norm(answer).split()
    return words[0] if words and words[0] in ("si", "no") else None


def parse_dt(value) -> datetime:
    return value if isinstance(value, datetime) else datetime.fromisoformat(str(value))


def probe_time(session_date, cfg: Stage6Config) -> datetime:
    """La sonda se hace en una sesión NUEVA posterior a after_session (tiempo simulado)."""
    return parse_dt(session_date) + timedelta(hours=cfg.probe_offset_hours)


def session_transcript(session: dict) -> str:
    lines = []
    for t in session["turns"]:
        lines.append(f"Usuario: {t['user']}")
        if t.get("assistant"):
            lines.append(f"Asistente: {t['assistant']}")
    return "\n".join(lines)


# --- el almacén --------------------------------------------------------------------------------------
class MemoryStore:
    """Registros + vectores + log de operaciones. Nada se borra: se invalida.

    `embed(list[str]) -> np.ndarray` devuelve vectores normalizados (coseno = producto punto).
    `now()` da el reloj real para created_at (inyectable en las pruebas).
    """

    def __init__(self, scenario_id: str, embed: Callable, now: Callable[[], datetime] | None = None):
        self.scenario_id = scenario_id
        self.embed = embed
        self.now = now or (lambda: datetime.now().replace(microsecond=0))
        self.records: list[MemoryRecord] = []
        self.vectors: dict[str, object] = {}
        self.oplog: list[dict] = []

    # consultas
    def get(self, memory_id: str) -> MemoryRecord | None:
        return next((r for r in self.records if r.memory_id == memory_id), None)

    def valid(self) -> list[MemoryRecord]:
        return [r for r in self.records if r.valid_to is None]

    def subjects(self) -> list[str]:
        return sorted({r.subject for r in self.valid()})

    def similar(self, text: str, subject: str, k: int, threshold: float) -> list[tuple[MemoryRecord, float]]:
        """Recuerdos VIGENTES parecidos: mismo subject, o similitud >= threshold."""
        current = self.valid()
        if not current:
            return []
        q = self.embed([text])[0]
        scored = [(r, float(self.vectors[r.memory_id] @ q)) for r in current]
        keep = [(r, s) for r, s in scored if s >= threshold or r.subject == subject]
        return sorted(keep, key=lambda x: (-(x[0].subject == subject), -x[1]))[:k]

    # escrituras
    def _new_id(self, session_id: str) -> str:
        return f"{self.scenario_id}-{session_id}-{sum(r.source_session == session_id for r in self.records) + 1:02d}"

    def add(self, *, subject, kind, text, importance, session_id, session_date, supersedes=None) -> MemoryRecord:
        rec = MemoryRecord(memory_id=self._new_id(session_id), kind=kind, subject=subject, text=text,
                           created_at=self.now(), valid_from=parse_dt(session_date),
                           source_session=session_id, importance=importance, supersedes=supersedes)
        self.records.append(rec)
        self.vectors[rec.memory_id] = self.embed([text])[0]
        return rec

    def invalidate(self, memory_id: str, session_date) -> MemoryRecord:
        rec = self.get(memory_id)
        if rec is None or rec.valid_to is not None:
            raise KeyError(memory_id)
        rec.valid_to = parse_dt(session_date)
        return rec

    def log(self, **op) -> dict:
        op = {"op_n": len(self.oplog) + 1, "scenario_id": self.scenario_id, **op}
        self.oplog.append(op)
        return op

    def dump(self) -> list[dict]:
        return [r.model_dump(mode="json") for r in self.records]


def write_session(store: MemoryStore, session: dict, call: Callable, cfg: Stage6Config) -> dict:
    """El camino de escritura para una sesión. `call(messages, schema, num_predict)` devuelve
    (objeto o None, ChatResult o None, error). Devuelve el costo de la sesión."""
    sid, sdate = session["session_id"], session["date"]
    stats = {"session_id": sid, "llm_calls": 0, "prompt_tokens": 0, "output_tokens": 0,
             "ops": {"ADD": 0, "UPDATE": 0, "NOOP": 0, "INVALIDATE": 0}, "errors": 0,
             "think_leaks": 0, "facts": 0, "added_without_llm": 0}

    def account(r):
        stats["llm_calls"] += 1
        if r is not None:
            stats["prompt_tokens"] += r.prompt_tokens or 0
            stats["output_tokens"] += r.output_tokens or 0
            stats["think_leaks"] += int(bool(r.think_leak))

    messages = [{"role": "system", "content": SYSTEM_EXTRACT},
                {"role": "user", "content": (
                    f"Fecha de la sesión: {sdate}\n"
                    f"Subjects ya guardados: {', '.join(store.subjects()) or '(ninguno)'}\n\n"
                    f"TRANSCRIPCIÓN DE LA SESIÓN {sid}:\n{session_transcript(session)}")}]
    extraction, r, error = call(messages, Extraction, cfg.extract_num_predict)
    account(r)
    if error or extraction is None:
        stats["errors"] += 1
        store.log(session_id=sid, step="extract", error=error or "sin extracción")
        return stats
    store.log(session_id=sid, step="extract", n_facts=len(extraction.facts),
              facts=[f.model_dump() for f in extraction.facts], analysis_discarded=extraction.analysis)
    stats["facts"] = len(extraction.facts)

    for fact in extraction.facts:
        neighbours = store.similar(fact.text, fact.subject, cfg.write_candidates, cfg.write_similarity)
        # Los hechos de una misma extracción ya vienen separados por atributo: solo se comparan con
        # recuerdos de sesiones anteriores, o del mismo subject (decidido en dev: el decisor marcaba
        # NOOP la academia contra el precio recién guardado de la misma sesión, y se perdía).
        neighbours = [(n, s) for n, s in neighbours if n.source_session != sid or n.subject == fact.subject]
        base = {"session_id": sid, "step": "write", "candidate": fact.model_dump(),
                "neighbours": [{"memory_id": n.memory_id, "subject": n.subject, "text": n.text,
                                "similarity": round(s, 3)} for n, s in neighbours]}
        if not neighbours:
            rec = store.add(subject=fact.subject, kind=fact.kind, text=fact.text, importance=fact.importance,
                            session_id=sid, session_date=sdate)
            stats["ops"]["ADD"] += 1
            stats["added_without_llm"] += 1
            store.log(**base, decision="ADD", via="sin recuerdos parecidos (sin LLM)", new_id=rec.memory_id)
            continue
        listing = "\n".join(f"- memory_id={n.memory_id} | subject={n.subject} | vigente desde "
                            f"{n.valid_from.date()} | {n.text}" for n, _ in neighbours)
        dmsgs = [{"role": "system", "content": SYSTEM_DECIDE},
                 {"role": "user", "content": (
                     f"Fecha de la sesión: {sdate}\n\nHECHO NUEVO (subject={fact.subject}): {fact.text}\n\n"
                     f"RECUERDOS GUARDADOS PARECIDOS:\n{listing}")}]
        decision, r, error = call(dmsgs, WriteDecision, cfg.decide_num_predict)
        account(r)
        ids = {n.memory_id for n, _ in neighbours}
        verdict, flag = (decision.decision if decision else "ADD"), None
        if error or decision is None:
            verdict, flag = "ADD", f"decisión fallida ({error}): se agrega"
            stats["errors"] += 1
        elif verdict in ("UPDATE", "INVALIDATE") and decision.target_id not in ids:
            verdict, flag = "ADD", f"target_id inválido ({decision.target_id}): se agrega"
            stats["errors"] += 1
        op = {**base, "decision": verdict, "llm_decision": decision.decision if decision else None,
              "target_id": decision.target_id if decision else None,
              "analysis_discarded": decision.analysis if decision else None, "flag": flag}
        if verdict == "ADD":
            rec = store.add(subject=fact.subject, kind=fact.kind, text=fact.text, importance=fact.importance,
                            session_id=sid, session_date=sdate)
            op["new_id"] = rec.memory_id
        elif verdict == "UPDATE":
            old = store.invalidate(decision.target_id, sdate)
            # Se guarda el texto del hecho (o el del decisor), nunca un análisis.
            rec = store.add(subject=old.subject, kind=fact.kind, text=(decision.text or fact.text),
                            importance=fact.importance, session_id=sid, session_date=sdate,
                            supersedes=old.memory_id)
            op["new_id"], op["invalidated_id"] = rec.memory_id, old.memory_id
        elif verdict == "INVALIDATE":
            store.invalidate(decision.target_id, sdate)
            op["invalidated_id"] = decision.target_id
        stats["ops"][verdict] += 1
        store.log(**op)
    return stats


# --- el camino de lectura ------------------------------------------------------------------------------
def score(recency: list[float], importance: list[float], relevance: list[float]) -> list[float]:
    """La combinación del enunciado: min-max por término y suma."""
    def minmax(xs):
        if not xs:
            return []
        lo, hi = min(xs), max(xs)
        if hi - lo < 1e-6:          # solo si es EXACTAMENTE constante
            return [0.0] * len(xs)
        return [(x - lo) / (hi - lo) for x in xs]
    return [a + b + c for a, b, c in zip(minmax(recency), minmax(importance), minmax(relevance))]


def memory_line(rec: MemoryRecord) -> str:
    if rec.valid_to is None:
        status = f"vigente desde {rec.valid_from.date()}, sesión {rec.source_session}"
    else:
        status = (f"YA NO VIGENTE: valió del {rec.valid_from.date()} al {rec.valid_to.date()}, "
                  f"sesión {rec.source_session}")
    return f"- [{status}] ({rec.kind}, {rec.subject}) {rec.text}"


def read_memories(store: MemoryStore, question: str, when: datetime, cfg: Stage6Config,
                  count: Callable[[str], int]) -> dict:
    """Top-k por relevancia + recencia + importancia, dentro de memory_budget.

    Incluye registros invalidados (marcados como tales): una pregunta "¿qué había dicho antes?"
    los necesita. La recencia usa valid_from (tiempo simulado), nunca el reloj real.
    """
    recs = [r for r in store.records if r.valid_from <= when]
    if not recs:
        return {"selected": [], "lines": [], "tokens": 0, "ranges": {}, "scores": []}
    q = store.embed([question])[0]
    relevance = [float(store.vectors[r.memory_id] @ q) for r in recs]
    hours = [max(0.0, (when - r.valid_from).total_seconds() / 3600) for r in recs]
    recency = [cfg.recency_decay ** h for h in hours]
    importance = [float(r.importance) for r in recs]
    ranges = {name: (round(min(v), 4), round(max(v), 4))
              for name, v in (("relevance", relevance), ("recency", recency), ("importance", importance))}
    total = score(recency, importance, relevance)
    order = sorted(range(len(recs)), key=lambda i: (-total[i], recs[i].memory_id))[:cfg.memory_k]
    as_passages = [{"chunk_id": recs[i].memory_id, "doc_id": "MEM", "section_id": recs[i].memory_id,
                    "index_text": memory_line(recs[i]), "rec": recs[i]} for i in order]
    kept, _, _ = assemble(as_passages, cfg.memory_budget, count, render=lambda ps: "\n".join(p["index_text"] for p in ps))
    chosen = sorted((p["rec"] for p in kept), key=lambda r: (r.valid_from, r.memory_id))   # cronológico
    lines = [memory_line(r) for r in chosen]
    return {"selected": chosen, "lines": lines, "tokens": count("\n".join(lines)) if lines else 0,
            "ranges": ranges,
            "scores": [{"memory_id": recs[i].memory_id, "total": round(total[i], 4), "relevance": round(relevance[i], 4),
                        "recency": round(recency[i], 4), "importance": importance[i]} for i in order]}


# --- la memoria como filtro de la recuperación ----------------------------------------------------------
DOC_ID_RE = re.compile(r"\b[A-Z]{2,5}-[0-9][\w\-/]*")
DATE_RE = re.compile(r"\b(\d{4}-\d{2}-\d{2})\b")


def memory_filter(records: list[MemoryRecord], known_doc_ids: set[str], docs_by_date: dict[str, list[str]]) -> dict:
    """De los recuerdos VIGENTES: los doc_id del corpus que el usuario nombró, y las fechas que
    coinciden con metadata.date (la fecha de publicación registrada, no la vigencia)."""
    doc_ids, dates, sources = set(), set(), []
    for r in records:
        if r.valid_to is not None:
            continue
        found = {d for d in DOC_ID_RE.findall(r.text) if d in known_doc_ids}
        found_dates = {d for d in DATE_RE.findall(r.text) if d in docs_by_date}
        if found or found_dates:
            sources.append(r.memory_id)
        doc_ids |= found
        dates |= found_dates
    date_docs = sorted({d for x in dates for d in docs_by_date[x]})
    return {"doc_ids": sorted(doc_ids) or None, "dates": sorted(dates) or None,
            "date_doc_ids": date_docs or None, "source_records": sources}


def retrieve_filtered(retriever, query: str, allowed_doc_ids: set[str] | None) -> list[dict]:
    """El ranking BM25 de la Etapa 3 restringido a ciertos documentos.

    Sin filtro devuelve lo mismo que retriever.retrieve. Con filtro puntúa TODOS los chunks y se
    queda con los de esos documentos (si se filtrara el top-100 ya cortado, un documento con la
    respuesta en el puesto 101 desaparecería).
    """
    if not allowed_doc_ids:
        return retriever.retrieve(query)
    words = retriever.words(query)
    if not words:
        return []
    idx, sc = retriever.index.retrieve(retriever.tokenize([" ".join(words)]), k=len(retriever.chunks),
                                       show_progress=False)
    ranked = [{**retriever.chunks[int(i)], "score": float(s)} for i, s in zip(idx[0], sc[0])
              if float(s) > 0 and retriever.chunks[int(i)]["doc_id"] in allowed_doc_ids]
    return sorted(ranked, key=lambda c: (-c["score"], c["chunk_id"]))[:retriever.cfg.candidate_k]


# --- historial (reproducción) ----------------------------------------------------------------------------
def session_messages(session: dict) -> list[dict]:
    msgs = []
    for j, t in enumerate(session["turns"]):
        prefix = f"[Sesión {session['session_id']} · {session['date']}]\n" if j == 0 else ""
        msgs.append({"role": "user", "content": prefix + t["user"]})
        if t.get("assistant"):
            msgs.append({"role": "assistant", "content": t["assistant"]})
    return msgs


def replay_history(sessions: list[dict], budget: int | None, count: Callable[[str], int]) -> dict:
    """Las sesiones pasadas, enteras, de la más reciente hacia atrás, mientras entren en `budget`.

    budget=None: todas (el brazo de reproducción completa verifica después que entren en num_ctx).
    """
    kept, used, full = [], 0, 0
    sizes = {s["session_id"]: sum(count(f"{m['role']}: {m['content']}") for m in session_messages(s)) for s in sessions}
    full = sum(sizes.values())
    for s in reversed(sessions):
        if budget is not None and used + sizes[s["session_id"]] > budget:
            break
        kept.insert(0, s)
        used += sizes[s["session_id"]]
    return {"messages": [m for s in kept for m in session_messages(s)], "sessions": [s["session_id"] for s in kept],
            "tokens": used, "full_tokens": full, "session_tokens": sizes,
            "dropped": [s["session_id"] for s in sessions if s not in kept]}


# --- el prompt de una sonda ------------------------------------------------------------------------------
def probe_user_message(question: str, when: datetime, memory_lines: list[str] | None, passages: list[dict]) -> str:
    parts = [f"Fecha de hoy: {when.date()} (sesión nueva)."]
    if memory_lines is not None:
        parts.append("MEMORIA:\n" + ("\n".join(memory_lines) if memory_lines else "(no hay recuerdos guardados)"))
    parts.append("PASAJES:\n" + (render_passages(passages) if passages else "(no se entregó ningún pasaje)"))
    parts.append(f"PREGUNTA: {question}")
    return "\n\n".join(parts)


def build_probe_prompt(question: str, when: datetime, ranked: list[dict], cfg: Stage6Config,
                       count: Callable[[str], int], count_messages: Callable[[list[dict]], int | None],
                       memory_lines: list[str] | None = None, history: list[dict] | None = None) -> dict:
    """Sistema + historial + (memoria + pasajes + pregunta). Los pasajes se llenan hasta
    passage_budget con el ensamblador de la Etapa 4 (saltea y sigue); si el prompt completo no
    entra en el tope, se saca el último pasaje (nunca la memoria ni el historial: si eso no
    entra, la fila es 'no factible' y lo decide quien llama)."""
    delivered, skipped, ptoks = assemble(ranked, cfg.passage_budget, count) if ranked else ([], [], 0)
    while True:
        msgs = ([{"role": "system", "content": SYSTEM_PROBE}] + list(history or [])
                + [{"role": "user", "content": probe_user_message(question, when, memory_lines, delivered)}])
        expected = count_messages(msgs)
        if expected is None or expected <= cfg.prompt_limit or not delivered:
            break
        skipped.append(delivered.pop())
        ptoks = count(render_passages(delivered)) if delivered else 0
    return {"messages": msgs, "delivered": delivered, "passage_tokens": ptoks,
            "expected_prompt_tokens": expected, "fits": expected is None or expected <= cfg.prompt_limit}


# --- puntaje sin juez ------------------------------------------------------------------------------------
def score_probe(probe: dict, answer: GroundedAnswer | None, error: str | None, delivered: list[dict]) -> dict:
    gold = {(g["doc_id"], g.get("section_id")) for g in probe.get("gold") or []}
    m = {"answerable": probe["answerable"], "ability": probe["ability"],
         "gold_delivered": (any((p["doc_id"], p["section_id"]) in gold for p in delivered) if gold else None)}
    if error is not None or answer is None:
        m.update(outcome="error", error=error or "sin respuesta", correct=None)
        return m
    abstained = not answer.can_answer
    expected = probe.get("expected_verdict")
    verdict = verdict_from_answer(answer.answer)
    key_hit = contains_key(answer.answer, probe.get("answer_key")) if probe.get("answer_key") else None
    if not probe["answerable"]:
        correct = abstained
    elif expected:
        correct = (not abstained) and verdict == {"yes": "si", "no": "no"}[expected]
    else:
        correct = (not abstained) and bool(key_hit)
    m.update(outcome="abstained" if abstained else "answered", error=None, correct=bool(correct),
             key_hit=key_hit, verdict=verdict, expected_verdict=expected,
             cited=[c for c in answer.citations if c])
    return m


def evidence_recall(probe: dict, sessions_seen: list[str]) -> float | None:
    ev = probe.get("evidence_sessions") or []
    if not ev:
        return None
    return sum(e in set(sessions_seen) for e in ev) / len(ev)


# --- auditoría de lo guardado --------------------------------------------------------------------------
HEDGE_RE = re.compile(r"\b(probablemente|posiblemente|quizas|tal vez|parece|supongo|podria|seguramente|asumo|"
                      r"infiero|aparentemente|presumiblemente)\b")
NUM_RE = re.compile(r"\d+")


def audit_records(records: list[MemoryRecord], sessions: dict[str, dict], analyses: list[str],
                  fact_texts: list[str] = ()) -> list[dict]:
    """Chequeos automáticos de cada registro (la lectura humana va aparte):
    - think_tag: contiene <think> o </think>;
    - hedge: formulado como hipótesis ("probablemente", "parece"...);
    - analysis_leak: el texto es (parte de) un `analysis` que el modelo escribió;
    - unsaid_numbers: números que no aparecen en la transcripción de su sesión."""
    out = []
    norm_analyses = [_norm(a) for a in analyses if a]
    norm_facts = {_norm(t) for t in fact_texts if t}
    for r in records:
        text_n = _norm(r.text)
        source = _norm(session_transcript(sessions[r.source_session])) if r.source_session in sessions else ""
        nums = [n for n in NUM_RE.findall(text_n) if n not in NUM_RE.findall(source)]
        out.append({"memory_id": r.memory_id, "subject": r.subject, "text": r.text,
                    "valid": r.valid_to is None,
                    "think_tag": "<think>" in r.text or "</think>" in r.text,
                    "hedge": bool(HEDGE_RE.search(text_n)),
                    "analysis_leak": text_n not in norm_facts and any(text_n and (text_n in a) for a in norm_analyses),
                    "unsaid_numbers": nums})
    return out


def ability_table(rows: list[dict], arms: list[str], abilities: list[str]) -> list[dict]:
    """Conteos por habilidad y brazo ('3/4', no 75%). Los errores se cuentan aparte."""
    table = []
    for ab in abilities + ["total"]:
        line = {"ability": ab}
        for arm in arms:
            rs = [r for r in rows if r["arm"] == arm and (ab == "total" or r["metrics"]["ability"] == ab)]
            ok = sum(bool(r["metrics"]["correct"]) for r in rs if r["metrics"]["outcome"] != "error")
            err = sum(r["metrics"]["outcome"] == "error" for r in rs)
            line[arm] = f"{ok}/{len(rs)}" + (f" ({err} err)" if err else "")
        table.append(line)
    return table


# --- embedder de respaldo ---------------------------------------------------------------------------
def lexical_embedder(dim: int = 4096):
    """Bolsa de palabras con hashing, normalizada. NO es el embedder del proyecto (bge-m3): sirve
    para las pruebas sin GPU y como respaldo declarado si bge-m3 no se puede cargar."""
    import hashlib as _h
    import numpy as np

    def embed(texts):
        out = np.zeros((len(texts), dim), dtype=np.float32)
        for i, t in enumerate(texts):
            for w in _norm(t).split():
                if len(w) > 2:
                    out[i, int(_h.md5(w[:6].encode()).hexdigest(), 16) % dim] += 1.0
            n = float(np.linalg.norm(out[i]))
            if n:
                out[i] /= n
        return out
    return embed


def to_jsonable(x):
    return json.loads(json.dumps(x, ensure_ascii=False, default=str))


IMPLEMENTATION_SHA256 = 'cfd44c28cc055e6da6f78a3c2fee4b94142428904f6f178f063fd7d7523fa4cd'

In [ ]:
CFG = Stage6Config(model=MODEL)
check_reasoning_first(Extraction, WriteDecision, GroundedAnswer)
print(json.dumps(asdict(CFG), indent=2), '\nprompt_limit:', CFG.prompt_limit)

{
  "model": "qwen3:8b",
  "num_ctx": 8192,
  "num_predict": 700,
  "safety_margin": 64,
  "passage_budget": 1500,
  "memory_budget": 600,
  "history_budget": 3000,
  "memory_k": 6,
  "recency_decay": 0.995,
  "probe_offset_hours": 24,
  "write_candidates": 3,
  "write_similarity": 0.55,
  "extract_num_predict": 800,
  "decide_num_predict": 300,
  "sweep_fractions": [
    0.5,
    0.25
  ],
  "temperature": 0.0,
  "seed": 0
} 
prompt_limit: 7428


## 3. Insumos: escenarios congelados, corpus e índice de la Etapa 3

Se verifica la evaluación congelada contra el corpus. El índice es el que eligió la Etapa 3
(`stage3_selected_retriever.json`); si no está en `artifacts/`, se copia desde Drive, igual que en la
Etapa 5. BM25 se reconstruye con la misma clase del notebook 03 (misma normalización y stemming).

In [ ]:
import schema
import llm
from console import StageRun, print_table, verdict, clip, wrap
from store import ResultsLog, write_json, write_jsonl
from grounded_stage4 import canonical_hash, read_jsonl


def local_or_drive(rel: str) -> Path:
    """artifacts/full/<rel>; si falta y Drive está montado, se copia desde Drive (como la Etapa 5)."""
    local, remote = ART_FULL / rel, DRIVE_ART / 'full' / rel
    if not local.exists() and remote.exists():
        if remote.is_dir():
            shutil.copytree(remote, local, dirs_exist_ok=True)
        else:
            local.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(remote, local)
        print('copiado desde Drive:', rel)
    if not local.exists():
        raise FileNotFoundError(f'Falta {local} (ni en artifacts/ ni en Drive).')
    return local


local_or_drive('documents.jsonl')
schema.validate_documents(DOC_PATH).show().raise_if_failed()
EV = schema.load_eval(EVAL_DIR, documents_path=DOC_PATH)          # hashes de los 3 archivos + corpus
schema.validate_eval(EVAL_DIR, DOC_PATH).show().raise_if_failed()
FROZEN = json.loads((EVAL_DIR / 'FROZEN.json').read_text(encoding='utf-8'))

SCENARIOS = [s for s in read_jsonl(EVAL_DIR / 'scenarios.jsonl') if s['split'] == SPLIT]
assert SCENARIOS, f'No hay escenarios para {SPLIT}'
if SMOKE_TEST:   # §7.4: un escenario de dos sesiones con tres sondas
    s0 = dict(SCENARIOS[0])
    s0['sessions'] = s0['sessions'][:2]
    keep = {s['session_id'] for s in s0['sessions']}
    s0['probes'] = [p for p in s0['probes'] if p['after_session'] in keep][:3]
    SCENARIOS = [s0]

DOCS = {d['doc_id']: d for d in read_jsonl(DOC_PATH)}
KNOWN_DOC_IDS = set(DOCS)
DOCS_BY_DATE = {}
for d in DOCS.values():
    if d['metadata'].get('date'):
        DOCS_BY_DATE.setdefault(d['metadata']['date'], []).append(d['doc_id'])

# --- índice de la Etapa 3 ----------------------------------------------------------------------
# La elección de la Etapa 3: la copia de Drive tiene prioridad sobre la de git (puede estar más nueva).
_sel_drive = DRIVE_ART / 'full' / 'index' / 'stage3_selected_retriever.json'
selection_path = _sel_drive if _sel_drive.is_file() else local_or_drive('index/stage3_selected_retriever.json')
SELECTION = json.loads(selection_path.read_text(encoding='utf-8'))
if SELECTION['selected_arm'] != 'bm25':
    raise NotImplementedError(f"La Etapa 3 eligió {SELECTION['selected_arm']}; esta etapa reconstruye BM25. "
                              'Cargá ese recuperador con stage3_runtime.load_stage3.')
if SELECTION['corpus_hash'] != FROZEN['documents_sha256']:
    raise ValueError('El índice elegido en la Etapa 3 es de otro corpus.')


def find_index_dir():
    """El índice BM25 de la Etapa 3: STAGE3_INDEX_DIR, o el nombrado en la elección (en artifacts/ o en
    Drive), o cualquier índice stage3_* del mismo corpus con chunks.jsonl (en artifacts/, Drive o en
    cualquier carpeta de MyDrive hasta 4 niveles). Si hay varios, avisa y toma el más reciente."""
    if STAGE3_INDEX_DIR is not None:
        return Path(STAGE3_INDEX_DIR)
    name = Path(SELECTION['index_dir']).name
    for base in (ART_FULL / 'index', DRIVE_ART / 'full' / 'index'):
        if (base / name / 'chunks.jsonl').is_file():
            return base / name
    roots = [ART_FULL / 'index', DRIVE_ART / 'full' / 'index']
    found = []
    for root in roots:
        found += [p.parent for p in root.glob('stage3_*/chunks.jsonl')] if root.exists() else []
    mydrive = Path('/content/drive/MyDrive')
    if not found and mydrive.exists():
        for depth in ('*/', '*/*/', '*/*/*/', '*/*/*/*/'):
            found += [p.parent for p in mydrive.glob(depth + 'stage3_*/chunks.jsonl')]
    same_corpus = []
    for d in dict.fromkeys(found):
        try:
            if json.loads((d / 'manifest.json').read_text(encoding='utf-8')).get('corpus_hash') == SELECTION['corpus_hash']:
                same_corpus.append(d)
        except (OSError, ValueError):
            pass
    if not same_corpus:
        raise FileNotFoundError(
            f"No encontré el índice BM25 de la Etapa 3 ({name}) ni otro del mismo corpus, ni en artifacts/ ni en "
            "Drive. Si la Etapa 3 la corrió otra persona, el índice está en SU Drive: que comparta la carpeta "
            "projecto_rag/artifacts/full/index/" + name + " (y la agregues como acceso directo en tu MyDrive), o "
            "fijá STAGE3_INDEX_DIR con la ruta. Alternativa: correr notebooks/03_retrieval.ipynb (SMOKE_TEST=False) "
            "hasta la sección 9, que reconstruye y guarda el índice BM25.")
    same_corpus.sort(key=lambda d: (d / 'chunks.jsonl').stat().st_mtime)
    if len(same_corpus) > 1 or same_corpus[-1].name != name:
        print(f'AVISO: no está {name}; uso {same_corpus[-1]} (mismo corpus). Encontrados: {[str(d) for d in same_corpus]}')
    return same_corpus[-1]


INDEX_DIR = find_index_dir()
print('Índice de la Etapa 3:', INDEX_DIR)




INDEX_MANIFEST = json.loads((INDEX_DIR / 'manifest.json').read_text(encoding='utf-8'))
CHUNKS = read_jsonl(INDEX_DIR / 'chunks.jsonl')
if canonical_hash(CHUNKS) != INDEX_MANIFEST['chunks_hash']:
    raise ValueError('chunks.jsonl no coincide con el manifiesto del índice de la Etapa 3.')

# La misma clase BM25Retriever del notebook 03 (igual que la Etapa 5).
nb3 = json.loads(Path('notebooks/03_retrieval.ipynb').read_text(encoding='utf-8'))
cells3 = []
for c in nb3['cells']:
    if c['cell_type'] != 'code':
        continue
    src = ''.join(c['source'])
    try:
        names = {n.name for n in ast.parse(src).body if isinstance(n, ast.ClassDef)}
    except SyntaxError:
        continue
    if {'BM25Retriever', 'RetrievalConfig'} <= names:
        cells3.append(src)
if len(cells3) != 1:
    raise RuntimeError(f'Esperaba una celda con BM25Retriever en el notebook 03; hay {len(cells3)}.')
mod3 = types.ModuleType('stage3_bm25_embedded')
sys.modules['stage3_bm25_embedded'] = mod3
exec(compile(cells3[0], 'stage3_bm25_embedded', 'exec'), mod3.__dict__)
RETRIEVER = mod3.BM25Retriever(CHUNKS, mod3.RetrievalConfig(**INDEX_MANIFEST['config']))
STAGE3_CELL_SHA256 = hashlib.sha256(cells3[0].encode('utf-8')).hexdigest()

# Guarda: cada gold de las sondas tiene que estar en el índice (y un gold faltante, a propósito, se rechaza).
SECTIONS_IN_INDEX = {(c['doc_id'], c['section_id']) for c in CHUNKS}


def check_probe_gold(scenarios):
    missing = [(s['scenario_id'], p['probe_id'], g) for s in scenarios for p in s['probes']
               for g in p['gold'] if (g['doc_id'], g['section_id']) not in SECTIONS_IN_INDEX]
    if missing:
        raise ValueError(f'Gold de sondas ausente del índice: {missing}')


check_probe_gold(SCENARIOS)
try:
    check_probe_gold([{'scenario_id': 'guarda', 'probes': [{'probe_id': 'p0', 'gold': [
        {'doc_id': '__falta__', 'section_id': '__falta__#ART-1'}]}]}])
except ValueError:
    print('OK: un gold faltante se rechaza.')
else:
    raise AssertionError('No saltó la guarda de gold.')

pending = sum('pendiente' in p.get('notes', '').lower() for s in SCENARIOS for p in s['probes'])
print(f"Split {SPLIT} | escenarios: {[s['scenario_id'] for s in SCENARIOS]} | sondas: "
      f"{sum(len(s['probes']) for s in SCENARIOS)} | chunks: {len(CHUNKS)} | índice: {INDEX_DIR.name}")
if pending:
    print(f'Nota: {pending} sondas conservan "revisión humana pendiente" en notes (texto del borrador; '
          'la revisión está en eval/revision_manual.csv).')

documents (documents.jsonl): OK
   documents: 170
   sections: 7070
   characters: 10685267
   with version: 0
   with date: 170
eval set (eval): OK
   single-turn items: 30
   by split: {'dev': 7, 'test': 23}
   by category: {'topical': 15, 'metadata': 7, 'identifier': 5, 'out_of_corpus': 3}
   dialogues: 8
   dialogue turns: 32
   scenarios: 3
   probes by ability: {'extraction': 3, 'multi_session': 6, 'temporal': 3, 'knowledge_update': 9, 'preference': 3, 'abstention': 3}
   yes/no rows (expected_verdict): 12
AVISO: no está stage3_36a95c9c1551289f; uso /content/drive/MyDrive/projecto_rag/artifacts/full/index/stage3_88c7f9546829caef (mismo corpus). Encontrados: ['/content/drive/MyDrive/projecto_rag/artifacts/full/index/stage3_88c7f9546829caef']
Índice de la Etapa 3: /content/drive/MyDrive/projecto_rag/artifacts/full/index/stage3_88c7f9546829caef


DEBUG:bm25s:Building index from IDs objects


OK: un gold faltante se rechaza.
Split test | escenarios: ['m002', 'm003'] | sondas: 18 | chunks: 13630 | índice: stage3_88c7f9546829caef
Nota: 13 sondas conservan "revisión humana pendiente" en notes (texto del borrador; la revisión está en eval/revision_manual.csv).


## 4. El embedder de la memoria

El mismo de la Etapa 3 (`BAAI/bge-m3`, en la revisión fijada allí), en fp16 sobre la GPU: ocupa ~1 GB y
entra junto a `qwen3:8b`. Solo codifica recuerdos y preguntas (decenas de textos cortos).

In [ ]:
if EMBEDDER == 'bge-m3':
    import torch
    from sentence_transformers import SentenceTransformer
    EMB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    EMB_REVISION = SELECTION.get('dense_revision')
    _encoder = SentenceTransformer(SELECTION.get('dense_model', 'BAAI/bge-m3'), revision=EMB_REVISION,
                                   device=EMB_DEVICE)
    if EMB_DEVICE == 'cuda':
        _encoder = _encoder.half()
    _encoder.max_seq_length = 512

    def embed(texts):
        return _encoder.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True,
                               batch_size=16, show_progress_bar=False)
    EMBEDDER_ID = {'model': SELECTION.get('dense_model', 'BAAI/bge-m3'), 'revision': EMB_REVISION,
                   'device': EMB_DEVICE, 'max_seq_length': 512}
else:
    embed = lexical_embedder()
    EMBEDDER_ID = {'model': 'lexical-hashing-bow', 'revision': None}
    warnings.warn('Embedder de respaldo: no es el de la Etapa 3. Declararlo en el informe.')

_v = embed(['El usuario prefiere el correo electrónico.', 'Canal de contacto preferido', 'Precio de la campera'])
print('Embedder:', EMBEDDER_ID, '| similitudes de prueba:', round(float(_v[0] @ _v[1]), 3), round(float(_v[0] @ _v[2]), 3))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedder: {'model': 'BAAI/bge-m3', 'revision': '5617a9f61b028005a4858fdac845db406aefb181', 'device': 'cuda', 'max_seq_length': 512} | similitudes de prueba: 0.697 0.314


## 5. El generador

Mismo servidor, modelo, `num_ctx`, `think=False` y temperatura 0 que las Etapas 4 y 5. Cada llamada usa
un schema (extracción, decisión de escritura o respuesta), y un error o un `ContextOverflow` se registra
como su propia categoría.

In [ ]:
environment.gpu_orphans()
INFO = llm.setup(MODEL, num_ctx=CFG.num_ctx)
print(llm.placement(MODEL))
llm.use_cache(ART / 'llm_cache' / f"{MODEL.replace(':', '_').replace('/', '_')}.jsonl")


def count(text):
    return llm.count_tokens(text, MODEL)


def count_messages(messages):
    return llm.count_prompt_tokens(messages, MODEL)


def call(messages, out_schema, num_predict):
    """(objeto o None, ChatResult o None, error) -- ContextOverflow atrapado y registrado."""
    try:
        obj, r = llm.structured(messages, out_schema, model=MODEL, num_ctx=CFG.num_ctx, num_predict=num_predict,
                                temperature=CFG.temperature, seed=CFG.seed, think=False)
    except llm.ContextOverflow as e:
        return None, None, f'CONTEXT_OVERFLOW (client): {e}'
    return obj, r, r.error


if count('hola') is None:
    raise RuntimeError(f'{MODEL} no tiene tokenizer en llm.HF_TOKENIZERS: las guardas estarían apagadas.')

THE GPU IS ALREADY OCCUPIED by other processes: 6162 MiB.
Almost always an orphan from a previous runtime. Restarting does NOT clear it:
  Runtime -> Disconnect and delete runtime, then start again.
   model        qwen3:8b
   digest       500a1f067a9f
   server       0.34.4
   num_ctx      8192
   thinking     {'honors_think_false': True, 'leaks_think_tag': False, 'thinking_only': False, 'supports_thinking': True, 'answer': 'Paris'}
   key_order    {'order': 'declaration', 'reply': '{"zeta": "alpha", "alfa": "omega"}', 'error': None}
   calibration  {'template_minus_server': [-4, -4, -4], 'offset': -4, 'repeat_prompt_tokens': (295, 295), 'repeat_cached_tokens': (269, 294), 'prompt_eval_count_includes_cached': True}
   placement    qwen3:8b: 100% GPU · 6.3 GB · context 8192
qwen3:8b: 100% GPU · 6.3 GB · context 8192
LLM call cache: /content/projecto_rag/artifacts/full/llm_cache/qwen3_8b.jsonl (52 entries)


## 6. Declarar el régimen: ¿la reproducción entra en la ventana?

Para cada escenario, el historial que reproduce `full_replay` después de cada sesión, contra lo que queda
de la ventana una vez descontados el sistema, los pasajes (1.500), la pregunta y la reserva de salida.
Toda afirmación de "la memoria le gana / pierde a la reproducción" vale para este régimen.

In [ ]:
SYSTEM_TOKENS = count(SYSTEM_PROBE)
WINDOW_FOR_HISTORY = CFG.prompt_limit - SYSTEM_TOKENS - CFG.passage_budget - CFG.memory_budget - 150
REGIME = []
for sc in SCENARIOS:
    for k in range(len(sc['sessions'])):
        h = replay_history(sc['sessions'][:k + 1], None, count)
        REGIME.append({'scenario': sc['scenario_id'], 'after': sc['sessions'][k]['session_id'],
                       'history_tokens': h['tokens'], 'history_budget': CFG.history_budget,
                       'fits_budget': h['tokens'] <= CFG.history_budget,
                       'fits_window': h['tokens'] <= WINDOW_FOR_HISTORY})
print_table('Régimen: tokens de historial reproducido', ['history_tokens', 'history_budget', 'fits_budget', 'fits_window'],
            [(f"{r['scenario']} tras {r['after']}", r) for r in REGIME], smoke=SMOKE_TEST)
per_session = [t for sc in SCENARIOS for t in replay_history(sc['sessions'], None, count)['session_tokens'].values()]
MEAN_SESSION_TOKENS = sum(per_session) / len(per_session)
SESSIONS_TO_OVERFLOW = int(min(CFG.history_budget, WINDOW_FOR_HISTORY) // MEAN_SESSION_TOKENS)
print(f'Una sesión ocupa en promedio {MEAN_SESSION_TOKENS:.0f} tokens. Con num_ctx {CFG.num_ctx}, la ventana deja '
      f'~{WINDOW_FOR_HISTORY} tokens para historial (presupuesto declarado {CFG.history_budget}): la reproducción '
      f'completa dejaría de entrar a partir de ~{SESSIONS_TO_OVERFLOW} sesiones de este tamaño. '
      f'Nuestros escenarios tienen {max(len(s["sessions"]) for s in SCENARIOS)}: régimen "entra".')


Régimen: tokens de historial reproducido
--------------------------------------------------------------------------
             | history_tokens | history_budget | fits_budget | fits_window
--------------------------------------------------------------------------
m002 tras s1 |            205 |          3,000 | True        | True       
m002 tras s2 |            316 |          3,000 | True        | True       
m002 tras s3 |            390 |          3,000 | True        | True       
m003 tras s1 |            203 |          3,000 | True        | True       
m003 tras s2 |            343 |          3,000 | True        | True       
m003 tras s3 |            460 |          3,000 | True        | True       
--------------------------------------------------------------------------
Una sesión ocupa en promedio 142 tokens. Con num_ctx 8192, la ventana deja ~4812 tokens para historial (presupuesto declarado 3000): la reproducción completa dejaría de entrar a partir de ~21 sesiones de este

## 7. Escribir la memoria y responder las sondas

Se recorre cada escenario sesión por sesión. Después de cada sesión:

1. **Camino de escritura:** extraer hechos con un schema → recuperar los recuerdos vigentes parecidos →
   decidir ADD / UPDATE / NOOP / INVALIDATE (si no hay ninguno parecido, ADD sin consultar al LLM) →
   guardar `created_at`, `valid_from`, `valid_to`. Solo se guarda el `text` de cada hecho; el `analysis` se
   descarta (queda en el log como `analysis_discarded`, para auditar).
2. **Sondas con `after_session` igual a esa sesión**, en una sesión nueva 24 h después, con cada brazo.
   El almacén que ve la sonda es exactamente el que existía en ese momento.

Las filas se guardan a medida que se calculan (`ResultsLog`): si Colab se corta, volver a correr retoma, y
la caché del LLM repite el camino de escritura sin volver a generar.

In [ ]:
ARMS = ['no_memory', 'full_replay', 'memory', 'memory_filter']
SWEEP_ARMS = {f'replay_{int(f * 100)}': f for f in CFG.sweep_fractions} if RUN_SWEEP else {}
ALL_ARMS = ARMS + list(SWEEP_ARMS)
PROMPTS_SHA256 = canonical_hash({'extract': SYSTEM_EXTRACT, 'decide': SYSTEM_DECIDE, 'probe': SYSTEM_PROBE,
                                 'schemas': [m.model_json_schema() for m in (Extraction, WriteDecision, GroundedAnswer)]})
RUN_SPEC = {
    'stage': '06_memory', 'profile': PROFILE, 'split': SPLIT, 'config': asdict(CFG), 'eval': EV.key(),
    'corpus_hash': FROZEN['documents_sha256'], 'stage3_index': INDEX_DIR.name,
    'stage3_bm25_cell_sha256': STAGE3_CELL_SHA256, 'model': MODEL, 'model_digest': INFO.get('digest'),
    'ollama_server': INFO.get('server'), 'embedder': EMBEDDER_ID, 'prompts_sha256': PROMPTS_SHA256,
    'implementation_sha256': IMPLEMENTATION_SHA256, 'arms': ALL_ARMS,
    'scenarios': {s['scenario_id']: [p['probe_id'] for p in s['probes']] for s in SCENARIOS},
}
RUN_KEY = canonical_hash(RUN_SPEC)
RESULTS_DIR = ART / 'results' / f'stage6_{RUN_KEY[:16]}'
MEMORY_DIR = ART / 'memory' / f'stage6_{RUN_KEY[:16]}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MEMORY_DIR.mkdir(parents=True, exist_ok=True)
write_json(RESULTS_DIR / 'run_spec.json', RUN_SPEC)
LOG = ResultsLog(RESULTS_DIR / 'probes.jsonl', RUN_SPEC)
run = StageRun.start('06_memory', ART, profile=PROFILE, config=asdict(CFG))
started_at = time.strftime('%Y-%m-%d %H:%M:%S')
n_probes = sum(len(s['probes']) for s in SCENARIOS)
print(f'Sondas: {n_probes} × {len(ALL_ARMS)} brazos = {n_probes * len(ALL_ARMS)} respuestas, más el camino de '
      f'escritura (~1 extracción por sesión + 1 decisión por hecho con recuerdos parecidos). '
      f'~{(n_probes * len(ALL_ARMS) + 6 * sum(len(s["sessions"]) for s in SCENARIOS)) * 12 / 60:.0f} min en una T4 '
      '(mucho menos si está en caché).')

STORES, WRITE_STATS, FILTER_ROWS, READ_TRACE = {}, [], [], []
t_start = time.time()
for sc in SCENARIOS:
    sid_ = sc['scenario_id']
    store = MemoryStore(sid_, embed)
    STORES[sid_] = store
    full_history_tokens = replay_history(sc['sessions'], None, count)['tokens']
    for k, session in enumerate(sc['sessions']):
        st = write_session(store, session, call, CFG)
        WRITE_STATS.append({'scenario': sid_, **st})
        write_jsonl(MEMORY_DIR / sid_ / f'records_after_{session["session_id"]}.jsonl', store.dump())
        print(f"[{sid_} {session['session_id']}] escritura: {st['ops']} · llamadas {st['llm_calls']} · "
              f"tokens {st['prompt_tokens']}+{st['output_tokens']} · errores {st['errors']}")
        past = sc['sessions'][:k + 1]
        for probe in [p for p in sc['probes'] if p['after_session'] == session['session_id']]:
            item_id = f"{sid_}/{probe['probe_id']}"
            when = probe_time(session['date'], CFG)
            q = probe['question']
            ranked = RETRIEVER.retrieve(q)
            mem = read_memories(store, q, when, CFG, count)
            filt = memory_filter(store.records, KNOWN_DOC_IDS, DOCS_BY_DATE)
            ranked_doc = retrieve_filtered(RETRIEVER, q, set(filt['doc_ids'])) if filt['doc_ids'] else ranked
            READ_TRACE.append({'item_id': item_id, 'ranges': mem['ranges'], 'scores': mem['scores'],
                               'selected': [r.memory_id for r in mem['selected']], 'filter': filt})
            if probe['gold']:   # la memoria como filtro: medición de recuperación, sin LLM
                gold = {(g['doc_id'], g['section_id']) for g in probe['gold']}
                ranked_date = (retrieve_filtered(RETRIEVER, q, set(filt['date_doc_ids'])) if filt['date_doc_ids'] else ranked)
                row_f = {'item_id': item_id, 'filter': filt}
                for name, rk in (('sin filtro', ranked), ('doc_id de memoria', ranked_doc), ('fecha de memoria', ranked_date)):
                    dl, _, toks = assemble(rk, CFG.passage_budget, count) if rk else ([], [], 0)
                    row_f[name] = {'gold_delivered': any((p['doc_id'], p['section_id']) in gold for p in dl),
                                   'gold_rank': next((i for i, c in enumerate(rk, 1) if (c['doc_id'], c['section_id']) in gold), None),
                                   'n_candidates': len(rk), 'docs_delivered': sorted({p['doc_id'] for p in dl})}
                FILTER_ROWS.append(row_f)
            plans = {
                'no_memory': dict(ranked=ranked),
                'full_replay': dict(ranked=ranked, replay=replay_history(past, CFG.history_budget, count)),
                'memory': dict(ranked=ranked, memory=mem),
                'memory_filter': dict(ranked=ranked_doc, memory=mem),
            }
            for arm, frac in SWEEP_ARMS.items():
                plans[arm] = dict(ranked=ranked, replay=replay_history(past, int(frac * full_history_tokens), count))
            for arm in ALL_ARMS:
                if LOG.done(arm, item_id):
                    continue
                plan = plans[arm]
                rep, m_ = plan.get('replay'), plan.get('memory')
                prompt = build_probe_prompt(q, when, plan['ranked'], CFG, count, count_messages,
                                            memory_lines=m_['lines'] if m_ else None,
                                            history=rep['messages'] if rep else None)
                if not prompt['fits']:
                    answer, r, error = None, None, 'NOT_FEASIBLE: memoria/historial no entran en el tope'
                else:
                    answer, r, error = call(prompt['messages'], GroundedAnswer, CFG.num_predict)
                metrics = score_probe(probe, answer, error, prompt['delivered'])
                seen = (rep['sessions'] if rep else [rec.source_session for rec in m_['selected']] if m_ else [])
                LOG.write({
                    'stage': '06_memory', 'arm': arm, 'item_id': item_id, 'scenario_id': sid_,
                    'probe_id': probe['probe_id'], 'ability': probe['ability'], 'after_session': probe['after_session'],
                    'question': q, 'answer_key': probe.get('answer_key'), 'expected_verdict': probe.get('expected_verdict'),
                    'metrics': metrics, 'error': error,
                    'evidence_sessions': probe.get('evidence_sessions'),
                    'evidence_recall': evidence_recall(probe, seen), 'sessions_seen': sorted(set(seen)),
                    'memory_ids': [rec.memory_id for rec in m_['selected']] if m_ else None,
                    'memory_tokens': m_['tokens'] if m_ else 0,
                    'history_tokens': rep['tokens'] if rep else 0,
                    'history_sessions': rep['sessions'] if rep else None,
                    'history_dropped': rep['dropped'] if rep else None,
                    'passage_tokens': prompt['passage_tokens'],
                    'expected_prompt_tokens': prompt['expected_prompt_tokens'],
                    'delivered': [{'doc_id': p['doc_id'], 'section_id': p['section_id']} for p in prompt['delivered']],
                    'filter_doc_ids': filt['doc_ids'] if arm == 'memory_filter' else None,
                    'llm': r.log() if r is not None else None,
                    'output': answer.model_dump() if answer is not None else None,
                    'raw_text': r.text[:1500] if (r is not None and answer is None) else None,
                })
                print(f"   {item_id:<10} {arm:<14} {metrics['outcome']:<9} correct={metrics['correct']}"
                      + (f' ERROR {clip(error, 80)}' if error else ''))
    write_jsonl(MEMORY_DIR / sid_ / 'records.jsonl', store.dump())
    write_jsonl(MEMORY_DIR / sid_ / 'oplog.jsonl', [to_jsonable(o) for o in store.oplog])
print(f'Tiempo: {time.time() - t_start:.0f} s')

ITEM_ORDER = [f"{s['scenario_id']}/{p['probe_id']}" for s in SCENARIOS for p in s['probes']]
ROWS = sorted([r for r in LOG.rows() if r['item_id'] in ITEM_ORDER and r['arm'] in ALL_ARMS],
              key=lambda r: (ALL_ARMS.index(r['arm']), ITEM_ORDER.index(r['item_id'])))
assert len(ROWS) == len(ITEM_ORDER) * len(ALL_ARMS), 'faltan filas: volvé a correr la celda'

== 06_memory · profile full · report -> /content/projecto_rag/artifacts/full/run_report.json
Sondas: 18 × 6 brazos = 108 respuestas, más el camino de escritura (~1 extracción por sesión + 1 decisión por hecho con recuerdos parecidos). ~29 min en una T4 (mucho menos si está en caché).
[m002 s1] escritura: {'ADD': 7, 'UPDATE': 0, 'NOOP': 0, 'INVALIDATE': 0} · llamadas 1 · tokens 656+428 · errores 0
   m002/p1    no_memory      abstained correct=False
   m002/p1    full_replay    answered  correct=True
   m002/p1    memory         answered  correct=True
   m002/p1    memory_filter  answered  correct=True
   m002/p1    replay_50      abstained correct=False
   m002/p1    replay_25      abstained correct=False
   m002/p8    no_memory      answered  correct=False
   m002/p8    full_replay    answered  correct=True
   m002/p8    memory         answered  correct=True
   m002/p8    memory_filter  answered  correct=True
   m002/p8    replay_50      answered  correct=False
   m002/p8    replay_25

## 8. Exactitud por habilidad (conteos, no porcentajes)

Con 2 sondas por habilidad y escenario, un porcentaje esconde que cada sonda vale 50 puntos.
`knowledge_update` incluye el par p8/p9 (la misma pregunta antes y después del cambio): se reporta aparte
cuántos pares salen bien completos, porque responder "sí" o "no" siempre acierta la mitad.

In [ ]:
ABILITIES = [a for a in schema.ABILITIES if any(r['ability'] == a for r in ROWS)]
TABLE = ability_table(ROWS, ALL_ARMS, ABILITIES)
print_table(f'Etapa 6 · sondas correctas por habilidad ({SPLIT})', ALL_ARMS,
            [(t['ability'], t) for t in TABLE], smoke=SMOKE_TEST, name_header='habilidad')

PAIRS = {}
for arm in ALL_ARMS:
    ok = tot = 0
    for sc in SCENARIOS:
        byp = {r['probe_id']: r for r in ROWS if r['arm'] == arm and r['scenario_id'] == sc['scenario_id']}
        if 'p8' in byp and 'p9' in byp:
            tot += 1
            ok += bool(byp['p8']['metrics']['correct']) and bool(byp['p9']['metrics']['correct'])
    PAIRS[arm] = f'{ok}/{tot}'
print('Pares p8+p9 correctos (antes y después del cambio):', PAIRS)

COST = {}
for arm in ALL_ARMS:
    rs = [r for r in ROWS if r['arm'] == arm]
    ev = [r['evidence_recall'] for r in rs if r['evidence_recall'] is not None]
    COST[arm] = {
        'correct': f"{sum(bool(r['metrics']['correct']) for r in rs)}/{len(rs)}",
        'evidence_recall': round(sum(ev) / len(ev), 3) if ev else None,
        'mean_prompt_tokens': round(sum(r['expected_prompt_tokens'] or 0 for r in rs) / len(rs)),
        'mean_memory_tokens': round(sum(r['memory_tokens'] for r in rs) / len(rs)),
        'mean_history_tokens': round(sum(r['history_tokens'] for r in rs) / len(rs)),
        'errors': sum(r['metrics']['outcome'] == 'error' for r in rs),
    }
write_cost = {'llm_calls': sum(w['llm_calls'] for w in WRITE_STATS),
              'prompt_tokens': sum(w['prompt_tokens'] for w in WRITE_STATS),
              'output_tokens': sum(w['output_tokens'] for w in WRITE_STATS)}
print_table('Costo y recall de sesiones con evidencia, por brazo', list(COST['no_memory']),
            list(COST.items()), smoke=SMOKE_TEST)
print_table('Camino de escritura: costo por sesión', ['llm_calls', 'prompt_tokens', 'output_tokens', 'facts',
            'added_without_llm', 'errors'], [(f"{w['scenario']} {w['session_id']}", w) for w in WRITE_STATS],
            smoke=SMOKE_TEST)
n_sessions = len(WRITE_STATS)
print(f"Escritura total: {write_cost['llm_calls']} llamadas, {write_cost['prompt_tokens']} tokens de prompt "
      f"({write_cost['prompt_tokens'] / max(1, n_sessions):.0f} por sesión). Lectura: 0 llamadas extra al LLM "
      '(solo el embedder).')


Etapa 6 · sondas correctas por habilidad (test)
-------------------------------------------------------------------------------------------
habilidad        | no_memory | full_replay | memory | memory_filter | replay_50 | replay_25
-------------------------------------------------------------------------------------------
extraction       | 0/2       | 2/2         | 2/2    | 2/2           | 1/2       | 0/2      
multi_session    | 2/4       | 4/4         | 4/4    | 4/4           | 2/4       | 2/4      
temporal         | 0/2       | 2/2         | 2/2    | 2/2           | 2/2       | 0/2      
knowledge_update | 0/6       | 6/6         | 6/6    | 6/6           | 5/6       | 0/6      
abstention       | 2/2       | 2/2         | 2/2    | 2/2           | 2/2       | 2/2      
preference       | 0/2       | 2/2         | 2/2    | 2/2           | 2/2       | 1/2      
total            | 4/18      | 18/18       | 18/18  | 18/18         | 14/18     | 5/18     
-------------------------------

## 9. Una actualización de conocimiento de punta a punta

Para cada escenario: los registros cuyo subject tuvo un UPDATE o un INVALIDATE, con su vigencia, las
operaciones del log que los tocaron, y qué respondió cada brazo en las sondas de actualización y temporales.

In [ ]:
for sc in SCENARIOS:
    store = STORES[sc['scenario_id']]
    touched = {o.get('invalidated_id') for o in store.oplog if o.get('invalidated_id')}
    subjects = {store.get(t).subject for t in touched if store.get(t)}
    print(f"\n=== {sc['scenario_id']} · subjects actualizados: {sorted(subjects) or 'ninguno'}")
    for rec in store.records:
        if rec.subject in subjects:
            print(' ', memory_line(rec), f'[{rec.memory_id}' + (f' reemplaza {rec.supersedes}]' if rec.supersedes else ']'))
    for o in store.oplog:
        if o.get('step') == 'write' and (o.get('invalidated_id') or o.get('decision') in ('UPDATE', 'INVALIDATE', 'NOOP')):
            print(f"  op {o['op_n']} [{o['session_id']}] {o['decision']} (LLM: {o.get('llm_decision')}) "
                  f"{o['candidate']['text']!r} -> nuevo {o.get('new_id')} · invalida {o.get('invalidated_id')}")
    for r in ROWS:
        if r['scenario_id'] == sc['scenario_id'] and r['ability'] in ('knowledge_update', 'temporal') and r['arm'] in ARMS:
            out = r.get('output') or {}
            print(f"  {r['probe_id']} {r['arm']:<14} correct={r['metrics']['correct']} · {clip(r['question'], 60)} "
                  f"-> {clip(out.get('answer') or r['error'] or '', 90)}")


=== m002 · subjects actualizados: ['canal_contacto', 'formato_respuesta']
  - [YA NO VIGENTE: valió del 2026-09-01 al 2026-09-08, sesión s1] (procedural, canal_contacto) El usuario prefiere contactarse por teléfono. [m002-s1-04]
  - [YA NO VIGENTE: valió del 2026-09-01 al 2026-09-15, sesión s1] (procedural, formato_respuesta) El usuario prefirió que las respuestas se presentaran en un párrafo. [m002-s1-07]
  - [vigente desde 2026-09-08, sesión s2] (procedural, canal_contacto) El canal preferido del usuario pasa a ser el correo electrónico. [m002-s2-02 reemplaza m002-s1-04]
  - [vigente desde 2026-09-15, sesión s3] (procedural, formato_respuesta) El usuario solicitó que las respuestas se presenten en listas con viñetas en lugar de párrafos. [m002-s3-01 reemplaza m002-s1-07]
  op 11 [s2] UPDATE (LLM: UPDATE) 'El canal preferido del usuario pasa a ser el correo electrónico.' -> nuevo m002-s2-02 · invalida m002-s1-04
  op 13 [s3] UPDATE (LLM: UPDATE) 'El usuario solicitó que las respuesta

## 10. Auditoría de lo guardado

Chequeos automáticos de cada registro: bloques `<think>`, hipótesis formuladas como hechos ("probablemente",
"parece"), texto copiado de un `analysis`, y números que no aparecen en la transcripción de su sesión.
La lista completa de registros va abajo para leerla: los chequeos no reemplazan esa lectura.

In [ ]:
AUDIT = {}
for sc in SCENARIOS:
    store = STORES[sc['scenario_id']]
    analyses = [o.get('analysis_discarded') for o in store.oplog if o.get('analysis_discarded')]
    facts = [f['text'] for o in store.oplog if o.get('step') == 'extract' for f in o.get('facts') or []]
    AUDIT[sc['scenario_id']] = audit_records(store.records, {s['session_id']: s for s in sc['sessions']}, analyses, facts)
flat = [a for v in AUDIT.values() for a in v]
AUDIT_COUNTS = {'records': len(flat), 'valid': sum(a['valid'] for a in flat),
                'think_tag': sum(a['think_tag'] for a in flat), 'hedge': sum(a['hedge'] for a in flat),
                'analysis_leak': sum(a['analysis_leak'] for a in flat),
                'unsaid_numbers': sum(bool(a['unsaid_numbers']) for a in flat),
                'think_leaks_llm': llm.STATS.get('think_leaks', 0)}
print('Auditoría:', AUDIT_COUNTS)
for sid_, items in AUDIT.items():
    print(f'\n--- {sid_}')
    for a in items:
        flags = [k for k in ('think_tag', 'hedge', 'analysis_leak') if a[k]] + (
            [f"números no dichos {a['unsaid_numbers']}"] if a['unsaid_numbers'] else [])
        print(f"  {a['memory_id']} {'vigente ' if a['valid'] else 'invalid.'} ({a['subject']}) {a['text']}"
              + (f'   <-- {flags}' if flags else ''))

Auditoría: {'records': 22, 'valid': 17, 'think_tag': 0, 'hedge': 0, 'analysis_leak': 0, 'unsaid_numbers': 0, 'think_leaks_llm': 0}

--- m002
  m002-s1-01 vigente  (nombre_usuario) El usuario se llama Mateo.
  m002-s1-02 vigente  (tienda_compra) El usuario compró un videojuego en Tienda Ejemplo Beta.
  m002-s1-03 vigente  (precio_compra) El usuario pagó 85000 pesos por el videojuego.
  m002-s1-04 invalid. (canal_contacto) El usuario prefiere contactarse por teléfono.
  m002-s1-05 vigente  (norma_referencia) El usuario indicó usar el documento LEY-26043 como referencia normativa.
  m002-s1-06 vigente  (fecha_norma) El usuario mencionó que la fecha de publicación registrada de LEY-26043 es 2005-07-18.
  m002-s1-07 invalid. (formato_respuesta) El usuario prefirió que las respuestas se presentaran en un párrafo.
  m002-s2-01 vigente  (costo_envio) El usuario pagó 4500 pesos por el envío.
  m002-s2-02 vigente  (canal_contacto) El canal preferido del usuario pasa a ser el correo electrónico.


## 11. La memoria como filtro de la recuperación

En las sondas que necesitan el corpus, los hechos vigentes que coinciden con metadatos se convierten en
filtros: el `doc_id` que el usuario pidió usar y la fecha de publicación que dijo (`metadata.date`, que es
fecha de publicación, no de vigencia). Se mide sin LLM si la sección gold llega con y sin filtro. Ojo con
el caso contrario: un filtro que deja afuera justo el pasaje que permitiría decir "eso no está en tu norma".

In [ ]:
for f in FILTER_ROWS:
    print(f"\n{f['item_id']} · filtro desde {f['filter']['source_records']}: doc_ids={f['filter']['doc_ids']} "
          f"fechas={f['filter']['dates']} -> documentos con esa fecha: {f['filter']['date_doc_ids']}")
    for name in ('sin filtro', 'doc_id de memoria', 'fecha de memoria'):
        x = f[name]
        print(f"   {name:<18} gold entregado={x['gold_delivered']} · puesto del gold={x['gold_rank']} · "
              f"candidatos={x['n_candidates']} · documentos entregados={clip(', '.join(x['docs_delivered']), 90)}")
# El filtro se aplica en todas las sondas del brazo memory_filter (un sistema real no sabe de antemano
# cuáles necesitan el corpus). En las sondas sobre datos del usuario suele dejarlas sin pasajes:
emptied = [r['item_id'] for r in ROWS if r['arm'] == 'memory_filter' and r['filter_doc_ids'] and not r['delivered']]
emptied_gold = [i for i in emptied if any(p['gold'] for s in SCENARIOS for p in s['probes']
                                          if f"{s['scenario_id']}/{p['probe_id']}" == i)]
print(f'\nmemory_filter dejó sin pasajes {len(emptied)} sondas: {emptied}')
print(verdict(not emptied_gold, 'ninguna de ellas necesitaba el corpus (son sobre datos del usuario)',
              f'{emptied_gold} necesitaban el corpus: el filtro sacó justo la evidencia'))


m002/p7 · filtro desde ['m002-s1-05', 'm002-s1-06']: doc_ids=['LEY-26043'] fechas=['2005-07-18'] -> documentos con esa fecha: ['LEY-26043']
   sin filtro         gold entregado=True · puesto del gold=1 · candidatos=100 · documentos entregados=DEC-151-2022, LEY-26043, LEY-27642
   doc_id de memoria  gold entregado=True · puesto del gold=1 · candidatos=6 · documentos entregados=LEY-26043
   fecha de memoria   gold entregado=True · puesto del gold=1 · candidatos=6 · documentos entregados=LEY-26043

m003/p7 · filtro desde ['m003-s1-05', 'm003-s1-06']: doc_ids=['LEY-27521'] fechas=['2019-12-20'] -> documentos con esa fecha: ['DEC-50-2019', 'LEY-27521']
   sin filtro         gold entregado=True · puesto del gold=1 · candidatos=100 · documentos entregados=DEC-375-2021, DEC-92-1997, LEY-26994, LEY-27521
   doc_id de memoria  gold entregado=True · puesto del gold=1 · candidatos=8 · documentos entregados=LEY-27521
   fecha de memoria   gold entregado=True · puesto del gold=1 · candidatos=100 · 

## 12. La trampa del término casi constante

El rango de cada término antes de normalizar, en las primeras sondas. La recencia usa las fechas
simuladas: entre sesiones separadas por una semana, 0,995 por hora va de ~0,89 (un día) a ~0,39
(una semana): un rango real, no ruido del reloj estirado por el min-max.

In [ ]:
for t in READ_TRACE[:6]:
    print(t['item_id'], t['ranges'], '| elegidos:', t['selected'])
const = [t['item_id'] for t in READ_TRACE if t['ranges'] and any(abs(hi - lo) < 1e-6 for lo, hi in t['ranges'].values())]
print('Sondas con algún término constante (se anula, no se estira):', const or 'ninguna')

m002/p1 {'relevance': (0.3579, 0.5801), 'recency': (0.8867, 0.8867), 'importance': (8.0, 9.0)} | elegidos: ['m002-s1-01', 'm002-s1-02', 'm002-s1-03', 'm002-s1-04', 'm002-s1-05', 'm002-s1-07']
m002/p8 {'relevance': (0.334, 0.7778), 'recency': (0.8867, 0.8867), 'importance': (8.0, 9.0)} | elegidos: ['m002-s1-01', 'm002-s1-02', 'm002-s1-03', 'm002-s1-04', 'm002-s1-05', 'm002-s1-07']
m002/p2 {'relevance': (0.3618, 0.71), 'recency': (0.382, 0.8867), 'importance': (7.0, 9.0)} | elegidos: ['m002-s1-02', 'm002-s1-03', 'm002-s1-05', 'm002-s1-07', 'm002-s2-01', 'm002-s2-02']
m002/p3 {'relevance': (0.3452, 0.6714), 'recency': (0.382, 0.8867), 'importance': (7.0, 9.0)} | elegidos: ['m002-s1-02', 'm002-s1-04', 'm002-s1-05', 'm002-s1-07', 'm002-s2-01', 'm002-s2-02']
m002/p4 {'relevance': (0.3027, 0.731), 'recency': (0.382, 0.8867), 'importance': (7.0, 9.0)} | elegidos: ['m002-s1-02', 'm002-s1-03', 'm002-s1-04', 'm002-s1-05', 'm002-s2-01', 'm002-s2-02']
m002/p9 {'relevance': (0.334, 0.7778), 'recency

## 13. ¿Dónde deja de entrar la reproducción, y qué cambia?

Los brazos `replay_50` y `replay_25` reproducen con un presupuesto de historial fijo igual al 50% y al 25%
del historial completo de cada escenario: al principio entra todo y, a medida que se acumulan sesiones,
se pierden las más viejas enteras. Es lo que pasaría con num_ctx fijo y muchas más sesiones.

In [ ]:
if SWEEP_ARMS:
    print_table('Reproducción con presupuesto: correctas por habilidad', ['full_replay', *SWEEP_ARMS, 'memory'],
                [(t['ability'], t) for t in TABLE], smoke=SMOKE_TEST, name_header='habilidad')
    for arm in SWEEP_ARMS:
        lost = [(r['item_id'], r['history_dropped']) for r in ROWS if r['arm'] == arm and r['history_dropped']]
        print(f'{arm}: sondas con sesiones perdidas: {lost}')
else:
    print('Barrido desactivado (RUN_SWEEP=False).')


Reproducción con presupuesto: correctas por habilidad
---------------------------------------------------------------
habilidad        | full_replay | replay_50 | replay_25 | memory
---------------------------------------------------------------
extraction       | 2/2         | 1/2       | 0/2       | 2/2   
multi_session    | 4/4         | 2/4       | 2/4       | 4/4   
temporal         | 2/2         | 2/2       | 0/2       | 2/2   
knowledge_update | 6/6         | 5/6       | 0/6       | 6/6   
abstention       | 2/2         | 2/2       | 2/2       | 2/2   
preference       | 2/2         | 2/2       | 1/2       | 2/2   
total            | 18/18       | 14/18     | 5/18      | 18/18 
---------------------------------------------------------------
replay_50: sondas con sesiones perdidas: [('m002/p1', ['s1']), ('m002/p2', ['s1']), ('m002/p3', ['s1']), ('m002/p4', ['s1']), ('m002/p5', ['s1']), ('m002/p6', ['s1']), ('m002/p7', ['s1']), ('m002/p8', ['s1']), ('m002/p9', ['s1']), ('m003/p2'

## 14. Guardar tablas e informe de la corrida

In [ ]:
write_json(RESULTS_DIR / 'ability_table.json', {'table': TABLE, 'pairs_p8_p9': PAIRS, 'cost': COST,
                                                'write_cost': write_cost, 'write_stats': WRITE_STATS})
write_json(RESULTS_DIR / 'audit.json', {'counts': AUDIT_COUNTS, 'records': AUDIT})
write_json(RESULTS_DIR / 'filter_eval.json', FILTER_ROWS)
write_json(RESULTS_DIR / 'read_trace.json', READ_TRACE)
write_json(RESULTS_DIR / 'regime.json', {'rows': REGIME, 'mean_session_tokens': MEAN_SESSION_TOKENS,
                                         'window_for_history': WINDOW_FOR_HISTORY,
                                         'sessions_to_overflow': SESSIONS_TO_OVERFLOW})
with (RESULTS_DIR / 'ability_table.csv').open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['ability'] + ALL_ARMS)
    w.writeheader()
    w.writerows(TABLE)

md = [f'# Etapa 6 · memoria entre sesiones ({PROFILE}, split {SPLIT})', '',
      f"Modelo `{MODEL}` (digest {INFO.get('digest')}), num_ctx {CFG.num_ctx}, num_predict {CFG.num_predict}; "
      f'pasajes ≤ {CFG.passage_budget}, memoria ≤ {CFG.memory_budget} ({CFG.memory_k} recuerdos), '
      f"historial ≤ {CFG.history_budget}. Embedder {EMBEDDER_ID['model']}. Corrida `{RUN_KEY[:16]}`.", '']
if SMOKE_TEST:
    md += ['> SMOKE_TEST: estos números no significan nada; solo prueban que el código corre.', '']
md += ['## Sondas correctas por habilidad', '', '| habilidad | ' + ' | '.join(ALL_ARMS) + ' |',
       '|---|' + '---|' * len(ALL_ARMS)]
md += ['| ' + t['ability'] + ' | ' + ' | '.join(t[a] for a in ALL_ARMS) + ' |' for t in TABLE]
md += ['', f'Pares p8+p9 correctos: {PAIRS}.', '', '## Costo por brazo', '',
       '| brazo | correctas | recall de sesiones con evidencia | tokens de prompt (media) | memoria | historial | errores |',
       '|---|---|---|---|---|---|---|']
md += [f"| {a} | {c['correct']} | {c['evidence_recall']} | {c['mean_prompt_tokens']} | {c['mean_memory_tokens']} | "
       f"{c['mean_history_tokens']} | {c['errors']} |" for a, c in COST.items()]
md += ['', f"Camino de escritura: {write_cost['llm_calls']} llamadas y {write_cost['prompt_tokens']} tokens de prompt "
       f'en {n_sessions} sesiones.', '',
       f"Régimen: historial máximo {max(r['history_tokens'] for r in REGIME)} tokens; la reproducción completa "
       f'dejaría de entrar a partir de ~{SESSIONS_TO_OVERFLOW} sesiones de {MEAN_SESSION_TOKENS:.0f} tokens.', '',
       f'## Auditoría del almacén', '', f'{AUDIT_COUNTS}', '', '## Filtro por memoria', '']
for f in FILTER_ROWS:
    md.append(f"- {f['item_id']}: sin filtro {f['sin filtro']['gold_delivered']} (puesto {f['sin filtro']['gold_rank']}); "
              f"doc_id {f['filter']['doc_ids']} {f['doc_id de memoria']['gold_delivered']} (puesto {f['doc_id de memoria']['gold_rank']}); "
              f"fecha {f['filter']['dates']} -> {f['filter']['date_doc_ids']} {f['fecha de memoria']['gold_delivered']}.")
(RESULTS_DIR / 'stage6_report_tables.md').write_text('\n'.join(md) + '\n', encoding='utf-8')

report = {'stage': '06_memory', 'status': 'ok', 'profile': PROFILE, 'split': SPLIT, 'started_at': started_at,
          'finished_at': time.strftime('%Y-%m-%d %H:%M:%S'), 'run_key': RUN_KEY, 'run_spec': RUN_SPEC,
          'results_dir': str(RESULTS_DIR), 'memory_dir': str(MEMORY_DIR), 'llm_setup': INFO,
          'llm_stats': dict(llm.STATS), 'table': TABLE, 'pairs': PAIRS, 'cost': COST, 'write_cost': write_cost,
          'audit': AUDIT_COUNTS,
          'limitations': ['smoke is not a benchmark' if SMOKE_TEST else
                          f'{n_probes} probes: report counts, no significance claims',
                          'stage 5 loop reused without query rewriting (probe question as query)',
                          'audit flags are heuristic; read the records']}
write_json(RESULTS_DIR / 'run_report.json', report)
run.note('results_dir', str(RESULTS_DIR))
run.note('memory_dir', str(MEMORY_DIR))
run.finish()

# Si artifacts/ no está enlazado a Drive (como en la corrida de la Etapa 5), copiar resultados y memoria.
if environment.on_colab() and not environment.ARTIFACTS.is_symlink() and DRIVE_ART.exists():
    for src in (RESULTS_DIR, MEMORY_DIR):
        dst = DRIVE_ART / src.relative_to(environment.ARTIFACTS)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print('copiado a Drive:', dst)
print('Resultados:', RESULTS_DIR)
print('Memoria:', MEMORY_DIR)

== 06_memory finished: ok in 696.9 s · LLM calls 91
copiado a Drive: /content/drive/MyDrive/projecto_rag/artifacts/full/results/stage6_15fb7ba6fadd4e1f
copiado a Drive: /content/drive/MyDrive/projecto_rag/artifacts/full/memory/stage6_15fb7ba6fadd4e1f
Resultados: /content/projecto_rag/artifacts/full/results/stage6_15fb7ba6fadd4e1f
Memoria: /content/projecto_rag/artifacts/full/memory/stage6_15fb7ba6fadd4e1f
